# De procesos estocásticos a series de tiempo: del proceso a los datos

**Curso:** Procesos Estocásticos · Ingeniería Industrial · Universidad EIA  
**Sesión:** S10_1 — Sesión Puente 1  
**Bloque:** Puente entre Procesos Estocásticos y Forecasting  
**Tipo de notebook:** Contenido (no gestionado por nbgrader — actividades formativas, sin nota).  
**Duración:** 2 horas. Los tiempos sugeridos incluyen lectura, ejecución y discusión.

**Pregunta orientadora**

> ¿Qué cambia cuando dejamos de conocer el mecanismo probabilístico del sistema y solamente disponemos de su comportamiento observado a través del tiempo?

Hasta ahora partíamos del modelo para estudiar el sistema. Hoy vamos a cambiar el punto de partida:

$$
\text{modelo probabilístico conocido} \longrightarrow \text{comportamiento del sistema}
$$

$$
\text{datos observados} \longrightarrow \text{comprensión del sistema}.
$$

**Apertura: 4 min.** Piensa qué información perderías si recibieras solamente el histórico de una empresa.

**Objetivos de esta sesión** — cada actividad está etiquetada con el objetivo que refuerza:

- **O1.** Diferenciar conceptualmente un proceso estocástico de una realización particular del proceso.
- **O2.** Reconocer una serie de observaciones ordenadas temporalmente como datos generados por un proceso estocástico.
- **O3.** Comprender que un mismo mecanismo probabilístico puede producir múltiples realizaciones diferentes.
- **O4.** Identificar la diferencia entre analizar un sistema cuando se conoce su modelo probabilístico y analizarlo cuando solamente se dispone de datos observados.
- **O5.** Relacionar el análisis del sistema con la lógica del proyecto integrador del curso.

**Lectura de objetivos: 4 min.** Al terminar, debes poder explicar estas diferencias usando el inventario de cámaras, sin hacer cálculos extensos.

---

## 1. ¿Dónde estamos? — *(O4)*

**Tiempo sugerido: 5 min.**

En el bloque de cadenas de Markov conocíamos el mecanismo de evolución del sistema. Nuestra ruta fue:

$$
P \longrightarrow P^n \longrightarrow \pi \longrightarrow \text{comportamiento del sistema}.
$$

$P$ describe las transiciones de una semana a la siguiente; $P^n$, las transiciones en varios pasos; y $\pi$, las proporciones de largo plazo en la cadena finita irreducible que estudiamos. Para interpretar $\pi$ además como límite de las probabilidades de estado usamos las condiciones vistas en S03.

Esa información describe probabilidades y comportamiento agregado: **no fija la secuencia exacta que observaremos**. Hoy seguimos con el mismo sistema, pero pasamos del mecanismo conocido a una historia particular de lo que ocurrió.

---

## 2. Recuperación del sistema de inventario — *(O4)*

**Tiempo sugerido: 7 min.**

Retomamos la tienda de cámaras. La demanda de la semana $t$ es

$$D_t\sim\operatorname{Poisson}(1),$$

con demandas independientes entre semanas, como en el modelo trabajado. El estado es

$$X_t=\text{inventario disponible al final de la semana }t,
\qquad X_t\in\{0,1,2,3\}.$$

Si $X_t=0$, se ordenan tres cámaras para iniciar la siguiente semana; en otro caso no se realiza pedido. Las ventas están limitadas por las unidades disponibles: el inventario final no puede ser negativo.

La matriz que ya conocemos, con filas y columnas en el orden $0,1,2,3$, es

$$
P=\begin{pmatrix}
0.0803&0.1839&0.3679&0.3679\\
0.6321&0.3679&0&0\\
0.2642&0.3679&0.3679&0\\
0.0803&0.1839&0.3679&0.3679
\end{pmatrix}.
$$

**$P$ representa el mecanismo probabilístico de evolución del inventario.** Cada fila indica cómo se distribuye el estado de la próxima semana dado el estado actual. Por ejemplo, $p_{10}=0.6321$ significa que, si terminamos esta semana con una cámara, la probabilidad de terminar la siguiente sin cámaras es aproximadamente 63.21 %.

Usaremos directamente esta matriz, presentada a cuatro decimales, sin volver a deducirla.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=4, suppress=True)

estados = np.array([0, 1, 2, 3])
P = np.array([
    [0.0803, 0.1839, 0.3679, 0.3679],
    [0.6321, 0.3679, 0.0000, 0.0000],
    [0.2642, 0.3679, 0.3679, 0.0000],
    [0.0803, 0.1839, 0.3679, 0.3679],
])

---

## 3. Del modelo a una trayectoria — *(O1, O2)*

**Tiempo sugerido: 15 min.**

Supongamos que al inicio de nuestra observación conocemos $X_0=3$. Simularemos **100 semanas posteriores**, por lo que guardaremos 101 valores: el estado inicial y los estados de las semanas 1 a 100.

En cada paso, el programa consulta la fila del estado actual y sortea el siguiente estado con esas probabilidades. La semilla permite reproducir la simulación; no es una regla de reposición ni un parámetro del sistema.

In [ ]:
T = 100
x0 = 3
rng = np.random.default_rng(42)

x = np.zeros(T + 1, dtype=int)
x[0] = x0
for t in range(T):
    x[t + 1] = rng.choice(estados, p=P[x[t]])

semanas = np.arange(T + 1)
plt.figure(figsize=(11, 3.5))
plt.plot(semanas, x, "o-", markersize=3, linewidth=1)
plt.yticks(estados)
plt.ylim(-0.2, 3.2)
plt.xlabel("Semana t (0 = estado inicial)")
plt.ylabel("Inventario final (cámaras)")
plt.title("Una realización: 100 semanas con el mecanismo P")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Las marcas son observaciones al final de cada semana. Las líneas ayudan a seguir su orden: no representan mediciones dentro de la semana.

> **Proceso estocástico:** la familia de variables aleatorias $\{X_t\}_{t\geq0}$, junto con su estructura probabilística.  
> **Realización:** una secuencia particular de valores obtenidos al observar o simular el proceso.

$$\{X_t\}_{t\geq0}=\text{proceso estocástico}$$

$$x_0,x_1,\ldots,x_T=\text{una realización observada hasta }T.$$

La mayúscula y la minúscula tienen funciones diferentes:

| Notación | Qué representa en la tienda |
|---|---|
| $X_{20}$ | La variable aleatoria «inventario al final de la semana 20» |
| $x_{20}$ | El número concreto de cámaras registrado esa semana en esta realización |
| $\{X_t\}_{t\geq0}$ | La familia completa de variables aleatorias del inventario |
| $x_0,\ldots,x_{100}$ | Una historia particular de 100 semanas, incluido el estado inicial |

**Lectura en parejas:** localiza la semana 20 en el gráfico. Explica por qué conocer el valor observado allí no determina el inventario de la semana 20 en otra simulación. No confundas la curva que acabas de ver con todo el proceso.

---

## 4. Un proceso, muchas realizaciones — *(O1, O3)*

**Tiempo sugerido: 15 min.**

Conservemos tanto $P$ como $X_0=3$. Cambiaremos únicamente la semilla. Así comparamos historias producidas bajo el **mismo mecanismo y la misma condición inicial**.

La primera fila repetirá la trayectoria anterior. Las otras dos muestran alternativas; no son las semanas siguientes de la primera.

In [ ]:
semillas = [42, 123, 2026]
trayectorias = np.zeros((len(semillas), T + 1), dtype=int)

for k, semilla in enumerate(semillas):
    rng = np.random.default_rng(semilla)
    trayectorias[k, 0] = x0
    for t in range(T):
        actual = trayectorias[k, t]
        trayectorias[k, t + 1] = rng.choice(estados, p=P[actual])

fig, ejes = plt.subplots(3, 1, figsize=(11, 7), sharex=True, sharey=True)
for k, ax in enumerate(ejes):
    ax.plot(semanas, trayectorias[k], "o-", markersize=2.5, linewidth=1)
    ax.set_title(f"Realización {k + 1} — semilla {semillas[k]}")
    ax.set_ylabel("Cámaras")
    ax.set_yticks(estados)
    ax.set_ylim(-0.2, 3.2)
    ax.grid(alpha=0.25)
ejes[-1].set_xlabel("Semana t")
fig.suptitle("Un mismo proceso: P y estado inicial compartidos")
plt.tight_layout()
plt.show()

### Actividad 1 — ¿Cambió el proceso o la realización? *(O1, O3, formativa)*

Responde primero individualmente y después contrasta con un compañero.

**a)** Las curvas son distintas. ¿Significa que cada una tiene una matriz $P$ diferente?

**b)** ¿Qué permanece igual y qué cambia cuando modificamos la semilla?

**c)** Un compañero afirma: «El proceso es la primera curva; las otras son errores de simulación». ¿Cómo corregirías esa frase?

**d)** Si repetimos el código con la misma semilla, la misma matriz y el mismo estado inicial, ¿qué esperamos observar?

In [ ]:
# Escribe tus respuestas antes de revelar la explicación.
respuesta_1a = "..."
respuesta_1b = "..."
respuesta_1c = "..."
respuesta_1d = "..."

<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

**a)** No. Las tres se generaron con la misma matriz. El mecanismo asigna probabilidades a distintas historias, no una única secuencia obligatoria.

**b)** Permanecen iguales los estados, la unidad temporal, la condición inicial y las probabilidades de transición. Cambia la secuencia de sorteos y, en general, los valores observados. Semillas distintas no garantizan matemáticamente secuencias distintas; aquí permiten explorar alternativas.

**c)** Cada curva es una realización del proceso, observada hasta la semana 100. Ninguna agota todas las historias posibles ni su estructura probabilística.

**d)** La misma trayectoria en este entorno de ejecución. La reproducibilidad del programa permite repetir el ejemplo; no elimina la incertidumbre del modelo.

</details>

$$
\text{El proceso describe todas las trayectorias posibles; una realización es solamente una de ellas.}
$$

Más precisamente, el proceso también describe **las probabilidades** asociadas a esas trayectorias; no es solo una lista de caminos posibles.

---

## 5. Ahora eliminemos el modelo — *(O2, O4)*

**Tiempo sugerido: 10 min.**

Ahora imagina que llegas a la empresa y nadie te entrega $P$. Solo recibes el histórico semanal del inventario final. Sabes qué mide la columna y cuándo se registró cada dato, pero no te entregan el mecanismo probabilístico que lo produjo.

Para representar esta situación usaremos **los mismos datos de la primera simulación**. Son datos simulados para la clase, no registros de una empresa real. El cambio está en la información disponible para quien analiza: desde ahora, razona mirando el histórico y deja de consultar $P$.

La tabla muestra las primeras 12 semanas y el gráfico amplía las primeras 24. El histórico completo conserva las 100 semanas.

In [ ]:
# El histórico contiene las semanas 1 a 100; x0 queda fuera del registro.
semanas_observadas = np.arange(1, T + 1)
inventario_observado = x[1:].copy()

filas = ["| Semana | Inventario final (cámaras) |", "|---|---|"]
for semana, valor in zip(semanas_observadas[:12], inventario_observado[:12]):
    filas.append(f"| {semana} | {valor} |")
display(Markdown("\n".join(filas)))

plt.figure(figsize=(10, 3.5))
plt.plot(semanas_observadas[:24], inventario_observado[:24], "o-", linewidth=1)
plt.xticks(np.arange(1, 25))
plt.yticks(estados)
plt.ylim(-0.2, 3.2)
plt.xlabel("Semana de registro")
plt.ylabel("Inventario final (cámaras)")
plt.title("Histórico observado — detalle de las primeras 24 semanas")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

| Modelo conocido | Datos observados |
|---|---|
| Disponemos de las probabilidades de transición. | Disponemos de los estados que efectivamente ocurrieron. |
| Podemos generar otras historias con el mecanismo. | Tenemos una historia finita y buscamos comprender su estructura. |
| Interpretamos el comportamiento desde las reglas del modelo. | Necesitamos relacionar los registros con el funcionamiento del sistema. |

### Actividad 2 — Lo que los datos dicen y lo que falta *(O2, O4, formativa)*

Usando solamente la tabla y el gráfico, escribe **dos observaciones descriptivas** sobre lo ocurrido y **una pregunta sobre el mecanismo** que esos registros por sí solos no resuelven.

**Tu respuesta:** …

<details>
<summary><b>▶ Revelar una respuesta orientadora</b></summary>

Puedes localizar semanas concretas con inventario cero o semanas consecutivas con el mismo inventario. Son afirmaciones sobre esta historia. También puedes preguntar cuál es la probabilidad de agotarse desde cada estado, o qué regla explica una reposición.

Una trayectoria finita no revela automáticamente las probabilidades exactas ni identifica por sí sola todas las reglas del sistema. Ver un patrón invita a investigarlo; no basta para asegurar que siempre se repetirá.

</details>

---

## 6. De realización a serie de tiempo — *(O1, O2)*

**Tiempo sugerido: 8 min.**

El histórico que recibimos tiene la forma

$$x_1,x_2,\ldots,x_T,$$

una secuencia de observaciones **ordenadas temporalmente**. En este caso, cada observación es el inventario al final de una semana y $T=100$. El valor $x_0$ que fijamos para iniciar la simulación queda como condición inicial; el histórico comienza en la semana 1.

> **Una serie temporal observada puede interpretarse como una realización de un proceso estocástico.**

Cambió nuestra forma de mirar la secuencia: ahora es el dato disponible para el análisis. El proceso sigue siendo la familia de variables aleatorias; la serie observada contiene valores concretos. Llamarla serie de tiempo no exige que sus valores sean continuos: aquí son conteos enteros entre 0 y 3.

**Comprueba la lectura:** en la tabla de la sección anterior, identifica la variable registrada, su unidad de medida y su unidad temporal. Explica por qué una lista de inventarios sin semanas ni orden perdería parte de la información.

<details>
<summary><b>▶ Revelar respuesta</b></summary>

La variable es el inventario final, medido en cámaras y registrado semanalmente. La semana permite saber qué ocurrió antes y después; una lista sin ese orden conserva valores, pero pierde sus relaciones temporales.

</details>

---

## 7. El orden de los datos también contiene información — *(O2, O4)*

**Tiempo sugerido: 15 min.**

Considera estas dos secuencias cortas de inventario. Son **ejemplos construidos para comparar órdenes**, no nuevos registros de la simulación anterior. Ambas contienen exactamente los mismos valores y frecuencias; las dos respetan las transiciones permitidas por la política del inventario.

| Semana | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Secuencia A | 3 | 3 | 3 | 2 | 2 | 2 | 1 | 1 | 1 | 0 | 0 | 0 |
| Secuencia B | 3 | 2 | 1 | 0 | 3 | 2 | 1 | 0 | 3 | 2 | 1 | 0 |

### Actividad 3 — Los mismos valores, otra historia *(O2, O4, formativa)*

**a)** Cuenta cuántas veces aparece cada estado en A y en B. ¿Podrías distinguirlas si solo recibieras esa tabla de frecuencias?

**b)** ¿En cuál aparecen períodos consecutivos más largos con el mismo inventario?

**c)** Describe qué viene después de un cero cuando hay una observación siguiente. ¿Qué diferencia temporal ocultaba la tabla de frecuencias?

**d)** ¿Basta este ejemplo corto para asegurar que una empresa seguirá ese patrón en el futuro? Justifica.

**Tus respuestas:** …

<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

**a)** Cada estado aparece tres veces en cada secuencia. Las frecuencias son iguales; por sí solas no distinguen las dos historias.

**b)** En A cada valor permanece tres semanas consecutivas. En B el inventario cambia cada semana. Llamamos **persistencia**, de manera intuitiva, a la permanencia de un valor o situación durante observaciones consecutivas.

**c)** En A, los ceros con sucesor van seguidos de cero; en B van seguidos de tres. El último valor de cada secuencia no tiene sucesor observado. La posición temporal aporta información sobre qué valores aparecen juntos y en qué orden. Un cero seguido de cero es posible: aunque se reponga, la demanda de la semana siguiente puede agotar las tres cámaras.

**d)** No. Son secuencias cortas construidas para ilustrar una diferencia. No permiten asegurar una regularidad futura ni establecer por sí solas el mecanismo que genera los datos.

</details>

Hablamos intuitivamente de **dependencia temporal** cuando conocer lo ocurrido en un momento aporta información sobre lo que puede ocurrir después. La persistencia es una posible manifestación; la dependencia también puede aparecer cuando los estados tienden a cambiar siguiendo ciertas reglas.

$$
\boxed{\begin{gathered}
\text{En una serie de tiempo no solo importa qué valores observamos;}\\
\text{también importa el orden en que ocurren.}
\end{gathered}}
$$

**Conexión con Markov:** en este inventario sabemos que el estado actual aporta información cuando conocemos $P$: por ejemplo, desde 1 cámara es imposible terminar la próxima semana con 3, mientras que desde 0 sí es posible. Que las filas 0 y 3 coincidan no elimina la diferencia entre las demás filas. La nueva pregunta es cómo detectar esa estructura cuando solo tenemos datos.

¿Conocer el inventario al final de esta semana aporta información sobre el inventario que tendremos al final de la próxima?

---

## 8. ¿Qué está generando los datos que observamos? — *(O2, O4)*

**Tiempo sugerido: 10 min.**

Volvamos al sistema para interpretar lo observado. Dos variables distintas participan en su funcionamiento:

$$D_t=\text{demanda semanal},\qquad X_t=\text{inventario final}.$$

La demanda representa las cámaras solicitadas durante la semana. El inventario representa lo que queda al terminarla. No son dos nombres para la misma cantidad.

Para obtener el inventario final de la semana $t$, se parte del estado al cierre de la semana anterior, $X_{t-1}$. Si era cero, se inicia con tres cámaras; si era positivo, se inicia con ese inventario. Después se atiende la demanda hasta agotar las unidades disponibles.

$$
\boxed{\begin{gathered}
\text{Demanda} + \text{estado actual} + \text{política de reposición}\\
\longrightarrow \text{inventario observado}
\end{gathered}}
$$

### Actividad 4 — Una demanda, distintos inventarios *(O2, O4, formativa)*

En ambos escenarios la demanda de la próxima semana resulta ser una cámara.

**a)** Si esta semana termina con cero cámaras, ¿cuántas quedan al final de la próxima?

**b)** Si esta semana termina con una cámara, ¿cuántas quedan al final de la próxima?

**c)** Si el registro muestra inventario final cero, ¿puedes saber exactamente cuánta demanda hubo esa semana?

**Tus respuestas:** …

<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

**a)** Dos: se reponen tres y se vende una. **b)** Cero: no se repone y se vende la única disponible.

**c)** No. Incluso si supiéramos que se inició con tres cámaras, demandas de tres, cuatro o más dejarían inventario cero. El inventario final no registra toda la demanda.

La misma demanda produce inventarios finales diferentes según el estado previo y la política. Aunque la demanda se modele como independiente entre semanas, el inventario conserva una relación temporal a través del estado y las reglas de reposición.

</details>

$$\boxed{\text{Los datos que observamos son producidos por un sistema.}}$$

Los datos reflejan tanto **incertidumbre** como **reglas operativas y decisiones**. Antes de analizarlos necesitamos comprender qué mide cada registro y cómo funciona el sistema que lo genera.

---

## 9. Proyecto integrador: un mismo sistema, dos perspectivas — *(O5)*

**Tiempo sugerido: 10 min.**

El proyecto integrador seguirá este cambio de perspectiva sobre **un mismo caso**. Cada equipo recibirá un caso previamente estructurado por el profesor con:

- contexto del sistema;
- definición de $X_t$;
- estados;
- unidad temporal;
- matriz de transición $P$;
- reglas de funcionamiento;
- datos temporales asociados;
- una situación de decisión de Ingeniería Industrial.

**Todos los proyectos deberán utilizar una cadena de Markov proporcionada por el profesor. Los equipos no diseñarán las cadenas desde cero.**

La arquitectura común del proyecto será:

$$
\boxed{\text{Caso} \rightarrow \text{Cadena de Markov} \rightarrow \text{Datos observados} \rightarrow \text{Forecasting} \rightarrow \text{Decisión}}
$$

La cadena permite interpretar el sistema desde su mecanismo probabilístico. Los datos permiten estudiar lo que se observa a través del tiempo. Más adelante incorporaremos forecasting para apoyar la misma decisión de Ingeniería Industrial.

No buscamos entregar

$$\text{ejercicio de Markov}+\text{ejercicio de forecasting},$$

sino

$$\boxed{\text{un mismo sistema analizado desde dos perspectivas}.}$$

En el ejemplo de cámaras, la cadena describe el inventario y el histórico registra ese mismo inventario semanal. Una pregunta de decisión podría ser cómo utilizar la información sobre el riesgo de agotamiento y la evolución esperada de la demanda para apoyar las decisiones de abastecimiento y la revisión de la política de reposición. El vínculo debe explicarse con igual claridad en cada caso recibido, identificando qué registran sus datos y cómo se relacionan con los estados y reglas del sistema.

**Discusión de equipo:** explica en dos frases qué se perdería si el análisis de Markov y el análisis temporal trataran problemas operativos desconectados. Hoy presentamos la integración; todavía no seleccionamos ni ajustamos modelos de forecasting.

---

## 10. Hito 0 — Comprensión del caso y planteamiento de la integración — *(O4, O5)*

**Tiempo sugerido: 12 min.**

Este primer hito es **formativo y orientado a comprensión, no a cálculo**. Trabajen sobre el caso proporcionado por el profesor y respondan en lenguaje del sistema:

1. ¿Qué sistema representa el caso y cuál es su problema operativo?
2. ¿Qué representa $X_t$, cuál es la unidad temporal y qué significa cada estado?
3. ¿Qué información proporciona $P$? Interpreten al menos dos $p_{ij}$ en contexto.
4. ¿Qué variable o variables están registradas a través del tiempo?
5. ¿Cómo se relacionan la cadena de Markov y la serie temporal proporcionada?
6. ¿Qué decisión de Ingeniería Industrial podría apoyarse combinando posteriormente Markov y análisis temporal?

Para interpretar cada $p_{ij}$, indiquen **estado de origen, estado de destino, duración del paso y probabilidad**. Para conectar cadena y serie, precisen si el dato registra directamente el estado u otra cantidad asociada al funcionamiento del caso.

### Actividad 5 — Borrador del Hito 0 *(O4, O5, formativa)*

Usen estos campos para organizar sus respuestas. Si el caso todavía no ha sido entregado, ensayen en clase con el inventario de cámaras; el hito del equipo se desarrollará sobre su caso asignado.

| Elemento | Respuesta del equipo |
|---|---|
| Sistema y problema operativo | … |
| $X_t$, unidad temporal y significado de cada estado | … |
| Información de $P$ y dos transiciones interpretadas | … |
| Variables registradas a través del tiempo | … |
| Relación entre cadena y serie | … |
| Decisión que podría apoyarse posteriormente | … |

**Revisión entre equipos:** comprueben que las interpretaciones de $p_{ij}$ son probabilidades condicionales y que la decisión propuesta corresponde al mismo sistema del histórico.

No se solicitan todavía $P^n$, $\pi$, tiempos de primera pasada ni otros cálculos extensos del proyecto. Tampoco se pide construir modelos de forecasting.

$$\boxed{\text{Comprender} \rightarrow \text{analizar} \rightarrow \text{pronosticar} \rightarrow \text{decidir}}$$

<details>
<summary><b>▶ Revelar una guía con el inventario de cámaras</b></summary>

1. **Sistema y problema:** una tienda repone cámaras cuando el inventario se agota; interesa comprender los períodos sin disponibilidad para apoyar la planificación de inventarios.
2. **Estado y tiempo:** $X_t$ es el número de cámaras al final de la semana $t$. Los estados 0, 1, 2 y 3 indican cuántas quedan; un paso equivale a una semana.
3. **Mecanismo:** $P$ contiene probabilidades de transición de una semana a la siguiente. $p_{10}=0.6321$ indica aproximadamente 63.21 % de probabilidad de pasar de una cámara a cero. $p_{03}=0.3679$ indica aproximadamente 36.79 % de terminar la siguiente semana con tres, dado que esta terminó con cero y se activa la reposición.
4. **Registro:** en nuestro histórico se registra inventario final semanal, no demanda semanal.
5. **Relación:** el histórico contiene valores observados de la misma variable de estado que modela la cadena. En clase lo generamos mediante esa cadena.
6. **Decisión:** apoyar la planificación de la reposición para reducir períodos sin disponibilidad. Más adelante se integrarán los análisis; hoy solo se plantea la decisión, sin calcular ni recomendar una nueva política.

Esta guía ilustra el nivel de interpretación esperado. Cada equipo debe explicar el vínculo específico de su caso.

</details>

---

## 11. Cierre de la sesión — *(O1–O5)*

**Tiempo sugerido: 5 min.**

Partimos de un mecanismo conocido y observamos que puede producir muchas realizaciones. Después tratamos una de ellas como un histórico: una serie de valores ordenados temporalmente. El orden aporta información que las frecuencias por sí solas no conservan, y los valores reflejan tanto incertidumbre como reglas del sistema.

**Autoevaluación breve:** explica a un compañero la diferencia entre $X_t$ y $x_t$, por qué dos trayectorias distintas pueden provenir del mismo proceso y cómo se conecta esta idea con el Hito 0.

La perspectiva trabajada en Markov fue

$$\boxed{\text{Modelo probabilístico} \rightarrow \text{comportamiento del sistema}}.$$

La nueva perspectiva abre un espacio que todavía tenemos que aprender a recorrer:

$$\boxed{\text{Datos observados} \rightarrow ? \rightarrow \text{comportamiento futuro}}.$$

**Referencia de continuidad:** Ross, S. M. *Introduction to Probability Models*, capítulo 4 (cadenas de Markov); retoma también los conceptos y actividades de S01–S03.

Dejamos abierta para **S10_2** la siguiente pregunta:

Si solamente observamos $x_1,x_2,\ldots,x_T$, ¿cómo podemos determinar si el pasado contiene información útil sobre el futuro?